In [ ]:
# %% Deep learning - Section 21.198
#    Transfer learning with ResNet-18

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.

In [1]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2             as imageio
import torchvision
import torchvision.transforms as T

from torch.utils.data                 import DataLoader,TensorDataset,Dataset,Subset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [ ]:
# %% Use GPU

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


In [ ]:
# %% Data

# We'll try the ResNet on a dataset on which it was not trained (STL10). Since
# ResNet is trained for images in a specific range (not [-1,1]), the mean and
# std normalisation values in the transform need to change

# Transformations (vals can be found online)
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

# Reacll that .ToTensor() normalises to range [0,1]
transform = T.Compose([ T.ToTensor(),
                        T.Normalize(mean=mean,std=std) ])

# Import data and apply the transform
trainset = torchvision.datasets.STL10(root='./data', download=True, split='train', transform=transform)
testset  = torchvision.datasets.STL10(root='./data', download=True, split='test',  transform=transform)

# Convert into DataLoader objects
batchsize    = 32
train_loader = DataLoader(trainset,batch_size=batchsize,shuffle=True,drop_last=True)
test_loader  = DataLoader(testset, batch_size=256)


In [ ]:
# %% Inspect dataset

# Check out the shape of the datasets
print('Data shapes (train/test):')
print( trainset.data.shape )
print( testset.data.shape )

# Check out the range of pixel intensity values
print('\nData value range:')
print( (np.min(trainset.data),np.max(trainset.data)) )

# Check out the unique categories
print('\nData categories:')
print( trainset.classes )


In [ ]:
# %% Some apparent issues

# It looks like the images are not normalised !

# But by now you know the transform is applied when calling the DataLoaders ...
X,y = next(iter(train_loader))

# Try again
print('Data shapes (train/test):')
print( X.data.shape )

# See range of pixel intensity values (now normalised)
print('\nData value range:')
print( (torch.min(X.data),torch.max(X.data)) )


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(4,4,figsize=(phi*6,6))

for (i,ax) in enumerate(axs.flatten()):

    # Extract an image and transpose back to [32x32x3], for visualisation also
    # undo the normalisation
    pic = X.data[i].numpy().transpose((1,2,0))
    pic = pic-np.min(pic)
    pic = pic/np.max(pic)

    label = trainset.classes[y[i]]

    ax.imshow(pic)
    ax.text(0,0,label,ha='left',va='top',fontweight='bold',color='k',backgroundcolor='y')
    ax.axis('off')

plt.tight_layout()

plt.savefig('figure5_transfer_learning_resnet18.png')
plt.show()
files.download('figure5_transfer_learning_resnet18.png')


In [ ]:
# %% Import ResNet-18

resnet = torchvision.models.resnet18(pretrained=True)
print(resnet)


In [ ]:
# %% Model's summary

resnet.to(device);
summary(resnet,(3,96,96))


In [46]:
# %% Freeze layers and modify the model

# Freeze all layers
for p in resnet.parameters():
    p.requires_grad = False

# Change final layer
resnet.fc = nn.Linear(512,10)

# Ship to GPU
resnet.to(device);


In [47]:
# %% Define loss function and optimizer

loss_fun  = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(resnet.parameters(),lr=0.001,momentum=.9)


In [ ]:
# %% Train the model

# Takes ~4 mnis per epoch without GPU; takes ~2 min overall with GPU
numepochs = 10

# Preallocate variables
train_loss = []
test_loss  = []
train_acc  = []
test_acc   = []

# loop over epochs
for epochi in range(numepochs):

    # Compute time
    start = time.time()

    # Loop over training data batches
    batch_loss = []
    batch_acc  = []

    for X,y in train_loader:

        # Ship data to GPU
        X = X.to(device)
        y = y.to(device)

        # Forward propagation and loss
        yHat = resnet(X)
        loss = loss_fun(yHat,y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Loss and accuracy from current batch
        batch_loss.append( loss.item() )
        batch_acc.append( torch.mean((torch.argmax(yHat,axis=1) == y).float()).item() )

    # Average losses and accuracies across the batches
    train_loss.append( np.mean(batch_loss) )
    train_acc.append( 100*np.mean(batch_acc) )

    # Test performance (here done in batches)
    resnet.eval()

    batch_acc  = []
    batch_loss = []

    for X,y in test_loader:

        # Ship data to GPU
        X = X.to(device)
        y = y.to(device)

        # Forward propagation and loss
        with torch.no_grad():
            yHat = resnet(X)
            loss = loss_fun(yHat,y)

        # Loss and accuracy from current batch
        batch_loss.append(loss.item())
        batch_acc.append( torch.mean((torch.argmax(yHat,axis=1) == y).float()).item() )

    resnet.train()

    # Average losses and accuracies across the batches
    test_loss.append( np.mean(batch_loss) )
    test_acc.append( 100*np.mean(batch_acc) )

    # Print status update
    print(f'Finished epoch {epochi+1}/{numepochs}. Test accuracy = {test_acc[epochi]:.2f}%')
    print(f'Time elapsed: {(time.time()-start)/60:.2f} min\n')


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,2,figsize=(1.5*phi*5,5))

ax[0].plot(train_loss,'o-',label='Train')
ax[0].plot(test_loss,'o-',label='Test')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss (MSE)')
ax[0].set_title('Model loss')
ax[0].legend()

ax[1].plot(train_acc,'o-',label='Train')
ax[1].plot(test_acc,'o-',label='Test')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Accuracy (%)')
ax[1].set_title(f'Final model train/test accuracy: {train_acc[-1]:.2f}/{test_acc[-1]:.2f}%')
ax[1].legend()

plt.suptitle('Pretrained ResNet-18 on STL10 data',fontweight='bold',fontsize=14)

plt.savefig('figure6_transfer_learning_resnet18.png')
plt.show()
files.download('figure6_transfer_learning_resnet18.png')


In [ ]:
# %% Plotting

# Inspect a few random images
X,y = next(iter(test_loader))
X   = X.to(device)
y   = y.to(device)

resnet.eval()
predictions = torch.argmax(resnet(X),axis=1)

phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(4,4,figsize=(phi*6,6))

for (i,ax) in enumerate(axs.flatten()):

    # Extract an image and transpose back to [32x32x3], for visualisation also
    # undo the normalisation
    pic = X.data[i].cpu().numpy().transpose((1,2,0))
    pic = pic-np.min(pic)
    pic = pic/np.max(pic)

    ax.imshow(pic)

    label = trainset.classes[predictions[i]]
    truec = trainset.classes[y[i]]
    title = f'True: {truec} - Pred: {label}'

    # Title with color-coded accuracy
    titlecolor = 'g' if truec==label else 'r'
    ax.text(48,90,title,ha='center',va='top',fontweight='bold',color='k',backgroundcolor=titlecolor,fontsize=8)
    ax.axis('off')

plt.tight_layout()

plt.savefig('figure7_transfer_learning_resnet18.png')
plt.show()
files.download('figure7_transfer_learning_resnet18.png')


In [ ]:
# %% Exercise 1
#    Try re-downloading the resnet18, unfreeze the layers, and re-run. This means you'll be fine-tuning the entire
#    network instead of only the final prediction layer.

# Just turn .requires_grad=True; the performance improves remarkably, and most
# importantly it becomes more stable


In [23]:
# %% Exercise 2
#    Download an untrained resnet18. This is simply the architecture with random weights (you'll still need to replace
#    the final layer so it has 10 outputs). Train this model; how is the performance?

# As one would expect, the performance drops a lot, still way above chance
# though, and the most interesting feature is the remarkable overfitting to the
# training data

# Load a fresh instance
resnet = torchvision.models.resnet18(weights=None)


In [31]:
# %% Exercise 3
#    I used SGD as the backprop method. Try re-running the analysis using Adam. Does this help or hurt the train and
#    test performance?

# So, switched back to freezed parameters and pretrained model for comparison;
# the Adam optimizer doesn't really improve the performance, but it definitely
# looks more stable

# Model, freeze and change layers, and switch to Adam
resnet = torchvision.models.resnet18(pretrained=True)

for p in resnet.parameters():
    p.requires_grad = False

resnet.fc = nn.Linear(512,10)
resnet.to(device);

loss_fun  = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(resnet.parameters(),lr=0.001)


In [40]:
# %% Exercise 4
#    ~80% accuracy is pretty decent considering we didn't do anything to optimize the model. Looking through the model
#    metaparameters, what are some things you would try to change if you wanted to boost performance?

# Going for an easy fix and keep Adam (with smaller lr and weight decay),
# unfreeze everything, add some more epochs (n=30), and add a sprinkle of data
# augumentation; still the performance doesn't really grow more than ~89%

# Data augmentation
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

transform_train = T.Compose([ T.RandomResizedCrop(96, scale=(0.8,1.0)),
                              T.RandomHorizontalFlip(),
                              T.RandomAffine(degrees=10, shear=10),
                              T.ToTensor(),
                              T.Normalize(mean=mean,std=std) ])

transform_test  = T.Compose([ T.ToTensor(),
                              T.Normalize(mean=mean,std=std) ])

trainset = torchvision.datasets.STL10(root='./data', download=True, split='train', transform=transform_train)
testset  = torchvision.datasets.STL10(root='./data', download=True, split='test',  transform=transform_test)

batchsize    = 32
train_loader = DataLoader(trainset,batch_size=batchsize,shuffle=True,drop_last=True)
test_loader  = DataLoader(testset, batch_size=256)

# Model, freeze and change layers, and switch to Adam
resnet = torchvision.models.resnet18(pretrained=True)

for p in resnet.parameters():
    p.requires_grad = True

resnet.fc = nn.Linear(512,10)
resnet.to(device);

loss_fun  = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(resnet.parameters(),lr=1e-4,weight_decay=1e-4)


In [44]:
# %% Exercise 5
#    You've seen earlier in the course that data normalization is important. This is particularly so for pretrained
#    networks, because the weights are tuned to specific numerical ranges. But how important is the *exact* numerical
#    range? To find out, re-run the code but remove the normalization transform. Thus, the images now will be in the
#    range [0,1], which is overlapping with but smaller than (and non-negative) the range that the network is trained on.

# Switched back to all the original parameters

# Remove normalisation

transform = T.Compose([ T.ToTensor() ])

trainset = torchvision.datasets.STL10(root='./data', download=True, split='train', transform=transform)
testset  = torchvision.datasets.STL10(root='./data', download=True, split='test',  transform=transform)

batchsize    = 32
train_loader = DataLoader(trainset,batch_size=batchsize,shuffle=True,drop_last=True)
test_loader  = DataLoader(testset, batch_size=256)
